# Tendencias de Consumo
**Proyecto:** Reserva Inteligente de Restaurantes — Etapa 3  
**Análisis:** Productos más vendidos, ingresos por categoría y variación MoM  
**Fuente:** Data Warehouse Hive (`restaurant_dw`)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("tendencias_consumo_notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.shuffle.partitions", "8")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("USE restaurant_dw")
print("Spark version:", spark.version)

In [ ]:
# Cargar tablas del warehouse
fact_pedido      = spark.table("fact_pedido")
dim_tiempo       = spark.table("dim_tiempo")
dim_plato        = spark.table("dim_plato")
dim_restaurante  = spark.table("dim_restaurante")
dim_estado       = spark.table("dim_estado_pedido")

print(f"fact_pedido: {fact_pedido.count()} filas")
fact_pedido.printSchema()

## 1. Ingresos por mes y categoría

In [ ]:
df_mes_cat = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_plato,        fact_pedido["id_plato"]        == dim_plato["id"])
    .join(dim_restaurante,  fact_pedido["id_restaurante"]  == dim_restaurante["id"])
    .join(dim_estado,       fact_pedido["id_estado_pedido"] == dim_estado["id"])
    .filter(F.col("dim_estado_pedido.nombre") == "completado")
    .groupBy(
        dim_tiempo["anio"], dim_tiempo["mes"], dim_tiempo["nombre_mes"],
        dim_plato["categoria"],
        dim_restaurante["nombre"].alias("restaurante"),
    )
    .agg(
        F.countDistinct(fact_pedido["id_pedido_origen"]).alias("total_pedidos"),
        F.sum("cantidad").alias("unidades_vendidas"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
        F.round(F.avg("precio_unitario"), 2).alias("precio_promedio"),
    )
    .orderBy("anio", "mes", F.desc("ingresos"))
)

df_mes_cat.show(20, truncate=False)

## 2. Variación MoM (mes a mes) por categoría

In [ ]:
w = Window.partitionBy("categoria", "restaurante").orderBy("anio", "mes")

df_mom = (
    df_mes_cat
    .withColumn("ingresos_mes_anterior", F.lag("ingresos", 1).over(w))
    .withColumn("variacion_mom_pct",
        F.when(F.col("ingresos_mes_anterior") > 0,
            F.round(
                (F.col("ingresos") - F.col("ingresos_mes_anterior"))
                * 100.0 / F.col("ingresos_mes_anterior"), 2
            )
        ).otherwise(F.lit(None))
    )
    .drop("ingresos_mes_anterior")
)

df_mom.select(
    "anio", "mes", "nombre_mes", "categoria",
    "restaurante", "ingresos", "variacion_mom_pct"
).show(20, truncate=False)

## 3. Top 5 platos más vendidos por mes

In [ ]:
df_top_base = (
    fact_pedido
    .join(dim_tiempo,  "id_tiempo")
    .join(dim_plato,    fact_pedido["id_plato"] == dim_plato["id"])
    .join(dim_estado,   fact_pedido["id_estado_pedido"] == dim_estado["id"])
    .filter(F.col("dim_estado_pedido.nombre") == "completado")
    .groupBy(
        dim_tiempo["anio"], dim_tiempo["mes"], dim_tiempo["nombre_mes"],
        dim_plato["nombre"].alias("plato"),
        dim_plato["categoria"],
    )
    .agg(
        F.sum("cantidad").alias("unidades_vendidas"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
    )
)

w_rank = Window.partitionBy("anio", "mes").orderBy(F.desc("unidades_vendidas"))

df_top = (
    df_top_base
    .withColumn("rank_mes", F.rank().over(w_rank))
    .filter(F.col("rank_mes") <= 5)
    .orderBy("anio", "mes", "rank_mes")
)

df_top.show(30, truncate=False)

## 4. Guardar resultados en Hive

In [ ]:
df_mom.write.mode("overwrite").saveAsTable("restaurant_dw.resultado_tendencias_mes_categoria")
df_top.write.mode("overwrite").saveAsTable("restaurant_dw.resultado_top_platos_mes")

print("✅ Resultados guardados en Hive.")

spark.stop()